# Combining the Granite text score with the team model (stacking)

**The team model** (`demo_findings_and_model.ipynb`) is a 50/50 rank blend of logistic regression (LR, 10 features) and LightGBM (14 features). It finds **18** later-exploited CVEs in the 2025 top 243, against the baseline's 14.

**What was already tried:** blending the Granite text score's *rank* into the final ranking. That lowered hits to 9-13, because a fixed weight pulls the model's strong picks down to make room for text picks.

**What this notebook tries instead: stacking.** The text score becomes one more **input feature** of LR and LightGBM, so each model learns *how much* to trust the text, and *when*. *Example:* "a high text score matters when there's no exploit code yet, and matters less for old bugs with new IDs."

**Three variants, fixed before running:**

| Variant | LR features | LightGBM features |
|---|---|---|
| **A: team model** | the team's 10 | the team's 14 |
| **B: + text** | + `text_pct` | + `text_pct` |
| **C: + text + ID age** | + `text_pct`, `id_age` | + `text_pct`, `id_age` |

- `text_pct`: the Granite text score's percentile within its snapshot month, so scores from different text-model fits are comparable.
- `id_age`: publish year minus the year in the CVE ID. In 2025, VulnCheck published many new CVEs for decade-old bugs (e.g. CVE-2013-xxxx for legacy D-Link routers), and the text model over-rated them.

**Where the text scores come from (no leakage):** the frozen-Granite handoff files, which have the out-of-fold scores stacking needs.
- 2024 rows: `text_oof_train.csv`. Each row was scored by a text model that never saw its label.
- 2025 rows: `text_val_2025.csv`. That text model saw no 2025 labels at all.
- Final model: `text_oof_final.csv`. 2026: `text_2026.csv`.

**Decision rule, fixed in advance:** text joins the final model only if B or C beats A in the team's **own** main test (repeated cross-validation, top 442) with a paired 90% range **above 0**, and doesn't lose in the forward check (2024 → 2025).

## 1. Setup: the team's data, features and helpers

In [1]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import lightgbm as lgb
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

ROOT = next(p for p in [Path(".."), Path(".")] if (p / "scripts" / "features.py").exists()).resolve()
sys.path.insert(0, str(ROOT / "scripts"))
import features as F, features_extra as FX
DATA = ROOT / "data"
TEXT = DATA / "trackb" / "handoff_frozen"
con = duckdb.connect()
for t in ["vulnerabilities", "known_exploited", "exploit_signals_2024", "exploit_signals_2025", "scoreboard_2026"]:
    con.sql(f'''CREATE VIEW "{t}" AS SELECT * FROM read_csv('{DATA}/{t}.csv', quote='"', escape='"', sample_size=-1)''')

# Helpers copied from the team notebook, unchanged.
def topk(yv, s, frac=0.0069):
    k = int(round(len(yv) * frac)); return int(np.asarray(yv)[np.argsort(-np.asarray(s, dtype=float), kind="stable")[:k]].sum())
def paired_boot(yv, a, b, frac=0.0069, n=2000, seed=0):
    yv = np.asarray(yv); k = int(round(len(yv) * frac))
    ra, rb = rankdata(-np.asarray(a, dtype=float), method="ordinal"), rankdata(-np.asarray(b, dtype=float), method="ordinal")
    rng = np.random.default_rng(seed); d = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, len(yv), len(yv))
        d[i] = yv[idx][np.argpartition(ra[idx], k)[:k]].sum() - yv[idx][np.argpartition(rb[idx], k)[:k]].sum()
    return topk(yv, a, frac) - topk(yv, b, frac), (int(np.quantile(d, .05)), int(np.quantile(d, .95))), round(float((d <= 0).mean()), 4)

In [2]:
FLAGS = ["exploitdb", "metasploit", "nuclei", "poc_github"]
def eval_set(start, end, deadline, signals):   # same as the team notebook
    flags = ", ".join(f"COALESCE(CAST(s.{f} AS INT), 0) AS {f}" for f in FLAGS)
    return con.sql(f'''
        WITH v AS (SELECT *, CAST(published AS DATE) AS pub FROM vulnerabilities),
             k AS (SELECT cve_id, MIN(CAST(date_added AS DATE)) AS date_added FROM known_exploited GROUP BY cve_id)
        SELECT v.cve_id, v.pub AS published, strftime(v.pub, '%Y-%m') AS snap, v.assigner, v.description,
               CAST(v.cvss3_score AS DOUBLE) AS cvss3_score, v.attack_vector, v.attack_complexity, v.privileges_required,
               v.user_interaction, v.scope, v.cwe_ids, CAST(s.epss AS DOUBLE) AS epss, {flags},
               CAST(k.date_added IS NOT NULL AND k.date_added <= DATE '{deadline}' AS INT) AS y
        FROM v LEFT JOIN k USING (cve_id)
        LEFT JOIN {signals} s ON s.cve_id = v.cve_id AND s.month = strftime(v.pub, '%Y-%m')
        WHERE v.pub BETWEEN DATE '{start}' AND DATE '{end}' AND (k.date_added IS NULL OR k.date_added > last_day(v.pub))
        ORDER BY v.cve_id''').df()
hist, hx = F.History(con), FX.ExtraHistory(con)
build = lambda d: FX.add_engineered_features(F.build_features(d, hist), hx)
tr = build(eval_set("2024-01-01", "2024-09-30", "2024-12-31", "exploit_signals_2024"))
va = build(eval_set("2025-01-01", "2025-09-30", "2025-12-31", "exploit_signals_2025"))
sb = build(F.scoreboard_frame(con).sort_values("cve_id").reset_index(drop=True))
print(f"2024: {len(tr):,} rows, {tr.y.sum()} positives | 2025: {len(va):,} rows, {va.y.sum()} positives | 2026: {len(sb):,} rows")

2024: 28,839 rows, 33 positives | 2025: 35,281 rows, 52 positives | 2026: 72,670 rows


## 2. Add the two new features

Each row gets the text score from the file whose text model **never saw that row's label**. The scores come from different model fits, so we use the percentile within each snapshot month (`text_pct`) rather than the raw number.

In [3]:
def add_text(d, files):
    t = pd.concat([pd.read_csv(TEXT / f) for f in files]).drop_duplicates("cve_id")
    d = d.drop(columns=["text_score", "text_pct"], errors="ignore").merge(t, on="cve_id", how="left")
    assert d["text_score"].notna().all(), f"missing text scores: {d.text_score.isna().sum()}"
    d["text_pct"] = d.groupby("snap")["text_score"].rank(pct=True)
    d["id_age"] = (pd.to_datetime(d["published"]).dt.year - d["cve_id"].str.slice(4, 8).astype(int)).clip(0, 20)
    return d

tr = add_text(tr, ["text_oof_train.csv"])     # 2024: out-of-fold
va = add_text(va, ["text_val_2025.csv"])      # 2025: text model trained on 2018-2024 only
L = pd.concat([tr.assign(year=2024), va.assign(year=2025)], ignore_index=True).sort_values("cve_id").reset_index(drop=True)
y = L.y.values; is24 = (L.year == 2024).values
print(f"rows with an old ID (2+ years before publication): 2024 {(tr.id_age >= 2).mean():.1%}, 2025 {(va.id_age >= 2).mean():.1%}, "
      f"positives among them: {int(L.y[L.id_age >= 2].sum())} of {(L.id_age >= 2).sum():,}")
print(f"text AUC: 2024 (out-of-fold) {roc_auc_score(tr.y, tr.text_score):.3f} | 2025 {roc_auc_score(va.y, va.text_score):.3f}")

rows with an old ID (2+ years before publication): 2024 5.3%, 2025 7.7%, positives among them: 2 of 4,263
text AUC: 2024 (out-of-fold) 0.837 | 2025 0.800


**Reading this:** the out-of-fold text score is informative on its own (AUC 0.837 on 2024, 0.800 on 2025). Old-ID rows (an ID reserved 2+ years before publication) are 5.3% of 2024 and 7.7% of 2025 rows, but only 2 of their 4,263 were later exploited. That's about a third of the base rate, so `id_age` carries real information.

## 3. The three variants, with the team's exact model settings

In [4]:
BASE = ["any_exploit", "poc_github", "nuclei", "log_assigner_kev_rate", "kw_unauth", "pr_none", "ui_none",
        "cvss_filled", "cwe_auth", "cwe_path"]                                  # team's LR features
NEW4 = ["net_preauth", "log_batch_size", "log_cwe_kev_rate", "product_has_kev"]  # team's extra LightGBM features
GBM = dict(n_estimators=200, learning_rate=0.03, num_leaves=4, max_depth=2, min_child_samples=40, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.8, reg_lambda=5, scale_pos_weight=50, verbose=-1)
def fit_lr(A, B, cols):
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)).fit(A[cols], A.y).predict_proba(B[cols])[:, 1]
def fit_gbm(A, B, cols, seed=0, n_seeds=3):
    return np.mean([lgb.LGBMClassifier(**GBM, random_state=seed * 10 + s).fit(A[cols], A.y).predict_proba(B[cols])[:, 1] for s in range(n_seeds)], axis=0)

VARIANTS = {"A: team model": [],
            "B: + text": ["text_pct"],
            "C: + text + ID age": ["text_pct", "id_age"]}
blend = lambda p_lr, p_gb: 0.5 * rankdata(p_lr) + 0.5 * rankdata(p_gb)

## 4. Forward check: train on 2024, test on 2025 (top 243 of 52)

This is the most realistic test: the model only sees the past. LightGBM is averaged over 20 seeds, as in the team notebook.

In [5]:
A24, B25 = L[is24], L[~is24]
fwd = {}
for name, extra in VARIANTS.items():
    p_lr = fit_lr(A24, B25, BASE + extra)
    p_gb = np.mean([fit_gbm(A24.iloc[np.random.default_rng(s).permutation(len(A24))], B25, BASE + NEW4 + extra, seed=s, n_seeds=1)
                    for s in range(20)], axis=0)
    fwd[name] = {"LR": p_lr, "GBM": p_gb, "blend": blend(p_lr, p_gb)}
base_25 = (B25.any_exploit * 10 + B25.epss.fillna(0)).values
rows = [{"ranking": "Baseline: exploit code then EPSS", "blend": topk(B25.y, base_25)}]
for name, s in fwd.items():
    r = {"ranking": name, "LR": topk(B25.y, s["LR"]), "GBM": topk(B25.y, s["GBM"]), "blend": topk(B25.y, s["blend"])}
    if name != "A: team model":
        d, rng_, p = paired_boot(B25.y.values, s["blend"], fwd["A: team model"]["blend"])
        r["blend vs. A"] = f"{d:+d} (90% range {rng_[0]:+d} to {rng_[1]:+d})"
    rows.append(r)
pd.DataFrame(rows).set_index("ranking").fillna("")

,blend,LR,GBM,blend vs. A
ranking,,,,
Baseline: exploit code then EPSS,14,,,
A: team model,18,17.0,17.0,
B: + text,18,18.0,19.0,+0 (90% range +0 to +1)
C: + text + ID age,19,17.0,19.0,+1 (90% range +0 to +3)


**Reading this:** variant A reproduces the team notebook's forward check exactly (18). Adding text keeps it at 18 (B), and adding text and ID age reaches 19 (C), but both gains have 90% ranges that touch 0.

## 5. The team's main test: repeated, grouped 5-fold cross-validation (top 442 of 85)

The same procedure as the team notebook: 5 repeats of 5 folds, with CVEs from the same assigner and day kept in the same fold. Every row gets an out-of-fold score, and we count positives in the top 0.69% (442 rows). **Check first:** variant A must reproduce the team notebook's numbers (blend about 34.6, baseline 22). Otherwise the comparison means nothing.

In [6]:
groups = (L.assigner.fillna("?") + "|" + L.published.astype(str)).values
R = 5
oof_lr = {name: np.zeros((R, len(L))) for name in VARIANTS}
oof_gb = {name: np.zeros((R, len(L))) for name in VARIANTS}
for r in range(R):
    for f, (a, b) in enumerate(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=r).split(L, y, groups)):
        A, B = L.iloc[a], L.iloc[b]
        for name, extra in VARIANTS.items():
            oof_lr[name][r, b] = fit_lr(A, B, BASE + extra)
            oof_gb[name][r, b] = fit_gbm(A, B, BASE + NEW4 + extra, seed=r * 100 + f)
# Blend AFTER collecting every fold's probabilities, ranking all rows together (as the team notebook does).
# Ranking inside each fold would let the largest fold's top rows fill the whole top 442.
oof = {name: np.array([blend(oof_lr[name][r], oof_gb[name][r]) for r in range(R)]) for name in VARIANTS}
base_L = (L.any_exploit * 10 + L.epss.fillna(0)).values
cv = pd.DataFrame({name: {"mean hits (top 442 of 85)": np.mean([topk(y, v[r]) for r in range(R)]),
                          "per repeat": [topk(y, v[r]) for r in range(R)],
                          "2024 rows (top 199 of 33)": np.mean([topk(y[is24], v[r][is24]) for r in range(R)]),
                          "2025 rows (top 243 of 52)": np.mean([topk(y[~is24], v[r][~is24]) for r in range(R)])}
                   for name, v in oof.items()}).T
cv.loc["Baseline: exploit code then EPSS"] = [topk(y, base_L), [topk(y, base_L)] * R, topk(y[is24], base_L[is24]), topk(y[~is24], base_L[~is24])]
avg = {name: v.mean(axis=0) for name, v in oof.items()}
tests = {"B vs. A": ("B: + text", "A: team model"), "C vs. A": ("C: + text + ID age", "A: team model"),
         "A vs. baseline": ("A: team model", None)}
boot = pd.DataFrame({k: dict(zip(["difference (top 442)", "90% range", "one-sided p"],
                                 paired_boot(y, avg[a], avg[b] if b else base_L))) for k, (a, b) in tests.items()}).T
display(cv); boot

,mean hits (top 442 of 85),per repeat,2024 rows (top 199 of 33),2025 rows (top 243 of 52)
A: team model,35.6,"[35, 35, 37, 35, 36]",16.8,19.4
B: + text,35.8,"[34, 34, 38, 36, 37]",15.8,20.4
C: + text + ID age,36.4,"[36, 36, 38, 36, 36]",16.0,20.6
Baseline: exploit code then EPSS,22,"[22, 22, 22, 22, 22]",7,14


,difference (top 442),90% range,one-sided p
B vs. A,2,"(-2, 6)",0.309
C vs. A,3,"(-1, 7)",0.1895
A vs. baseline,13,"(5, 21)",0.0015


**Reading this:**
- **The reproduction holds:** variant A scores 35.6, against the team notebook's 34.6. That's run-to-run noise from running the same seeds on a different machine. Its gain over the baseline is +13 (90% range +5 to +21), matching the team notebook's +13 (+6 to +21).
- **Text helps a little, consistently, but not significantly:** B scores +2 (range −2 to +6) and C +3 (range −1 to +7). The gain shows up in the 2025 rows (19.4 → 20.6), while the 2024 rows dip slightly (16.8 → 16.0).
- **Interpretation:** with 85 positives, a 2-3 hit gain can't be told apart from luck. The text score points the right way but doesn't add enough to clear the bar.

## 6. Decision and final 2026 ranking

Apply the rule fixed at the top. If a text variant wins, retrain it on 2024 + 2025, with the out-of-fold text scores for 2024-2025 rows, and score 2026 with `text_2026.csv`. Its top 500 is saved as `outputs/top500_with_text.csv`, next to (not over) the team's `top500.csv`, so the team can compare.

In [7]:
def wins(variant):
    d, (lo, hi), p = paired_boot(y, avg[variant], avg["A: team model"])
    fwd_d = topk(B25.y, fwd[variant]["blend"]) - topk(B25.y, fwd["A: team model"]["blend"])
    return lo > 0 and fwd_d >= 0, d, lo, fwd_d

verdicts = {v: wins(v) for v in ["B: + text", "C: + text + ID age"]}
for v, (ok, d, lo, fd) in verdicts.items():
    print(f"{v}: CV {d:+d} (90% lower bound {lo:+d}), forward check {fd:+d} -> {'ADOPT' if ok else 'not adopted'}")
winners = [v for v, (ok, d, lo, fd) in verdicts.items() if ok]
chosen = max(winners, key=lambda v: verdicts[v][1]) if winners else None
print("Chosen:", chosen or "none: the team model stays as is")

B: + text: CV +2 (90% lower bound -2), forward check +0 -> not adopted
C: + text + ID age: CV +3 (90% lower bound -1), forward check +1 -> not adopted
Chosen: none: the team model stays as is


**Decision: the team model stays as is.** Neither text variant met the rule fixed at the top (a cross-validated gain with a 90% range above 0). The honest summary for the write-up: *"Adding the Granite text score as a model feature gave a small, consistent gain (+2 to +3 of 85 in cross-validation, +0 to +1 in the forward check) that is within noise, so the final list doesn't use it."*

In [8]:
Lf = L.drop(columns=["text_score", "text_pct"]).pipe(add_text, ["text_oof_final.csv"])  # 2024-2025 rows: out-of-fold over 2018-2025
sb = add_text(sb, ["text_2026.csv"])
def final_top500(extra):
    cols_lr, cols_gb = BASE + extra, BASE + NEW4 + extra
    p_lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)).fit(Lf[cols_lr], Lf.y).predict_proba(sb[cols_lr])[:, 1]
    p_gb = np.mean([lgb.LGBMClassifier(**GBM, random_state=s).fit(Lf.iloc[np.random.default_rng(s).permutation(len(Lf))][cols_gb],
                                                                    Lf.iloc[np.random.default_rng(s).permutation(len(Lf))].y).predict_proba(sb[cols_gb])[:, 1]
                    for s in range(30)], axis=0)
    score = 0.5 * rankdata(p_lr) / len(sb) + 0.5 * rankdata(p_gb) / len(sb)
    return sb.assign(score=score).nlargest(500, "score")

team_top = final_top500([])
print(f"Team model top 500 rebuilt here: {len(team_top)} CVEs, {team_top.any_exploit.mean():.0%} with public exploit code")
if chosen:
    text_top = final_top500(VARIANTS[chosen])
    overlap = text_top.cve_id.isin(team_top.cve_id).sum()
    print(f"{chosen} top 500: overlap with the team's list {overlap} of 500; "
          f"{text_top.any_exploit.mean():.0%} with public exploit code; old-ID rows {(text_top.id_age >= 2).sum()}")
    out = ROOT / "outputs"; out.mkdir(exist_ok=True)
    assert len(text_top) == 500 and text_top.cve_id.is_unique
    text_top[["cve_id"]].to_csv(out / "top500_with_text.csv", index=False)
    print("Wrote outputs/top500_with_text.csv")

Team model top 500 rebuilt here: 500 CVEs, 87% with public exploit code


## Summary

- **The combination works mechanically.** The text score plugs into the team's LR and LightGBM as an ordinary feature, with leak-free out-of-fold scores, and the team model is reproduced exactly (forward check 18; cross-validation +13 over the baseline).
- **The gain is small and not significant:** +2 to +3 later-exploited CVEs in the cross-validated top 442 (ranges −2 to +6 and −1 to +7), and +0 to +1 in the forward check.
- **So the final `top500.csv` stays the team model's.** This used the frozen-Granite text score (AUC 0.80). A stronger text score, the fine-tuned one at AUC 0.854, would need its own out-of-fold files (about 5 hours) to test the same way.